```text
╔══════════════════════════════════════╗
║          🟧 DAY 4 — 2 HRS            ║
╠══════════════════════════════════════╣
║                                      ║
║ 1. GENAI — 60 MIN                    ║
║                                      ║
║ □ Transformer Architecture           ║
║ □ Encoder                            ║
║ □ Decoder                            ║
║ □ Self-Attention → FFN → Output      ║
║                                      ║
║ 2. DSA — 45 MIN                      ║
║                                      ║
║ □ Sliding Window                     ║
║ □ 2 Easy problems                    ║
║                                      ║
║ 3. PRACTICE — 15 MIN                 ║
║                                      ║
║ □ Trace a Transformer flow           ║
║ □ GitHub progress                    ║
║                                      ║
╚══════════════════════════════════════╝
```



```text
=======================================================================
                      THE TRANSFORMER ARCHITECTURE
=======================================================================

         [ ENCODER ]                              [ DECODER ]
                                                       ▲
                                                       │
                                            [ Output Probabilities ]
                                                       ▲
                                                       │
                                                  [ Softmax ]
                                                       ▲
                                                       │
                                                  [ Linear ]
                                                       ▲
                                                       │
       ┌───────────────────┐                 ┌───────────────────┐
       │   [ Add & Norm ]  │                 │   [ Add & Norm ]  │
       │         ▲         │                 │         ▲         │
       │ [  Feed Forward ] │                 │ [  Feed Forward ] │
       │         ▲         │                 │         ▲         │
  Nx   │   [ Add & Norm ]  │            Nx   │   [ Add & Norm ]  │
       │         ▲         │                 │         ▲         │
       │ [  Multi-Head   ] │────────────────►│ [  Multi-Head   ] │ (Cross
       │ [   Attention   ] │                 │ [   Attention   ] │  Attention)
       └─────────▲─────────┘                 │         ▲         │
                 │                           │   [ Add & Norm ]  │
                 │                           │         ▲         │
                 │                           │ [ Masked Multi- ] │
                 │                           │ [ Head Attention] │
                 │                           └─────────▲─────────┘
                 │                                     │
           [ ⊕ ] ◄── [ Positional ]              [ ⊕ ] ◄── [ Positional ]
             ▲       [  Encoding  ]                ▲       [  Encoding  ]
             │                                     │
     [ Input Embeddings ]                  [ Output Embeddings ]
             ▲                                     ▲
             │                                     │
         [ Inputs ]                     [ Outputs (Shifted Right) ]

=======================================================================

```


<br>

**[  ENCODER  ]**

1. Inputs:
   The sequence starts as raw data (like words).

2. Input Embeddings:
   The raw inputs are converted into vectors (numbers).

3. Positional Encoding:
   Because the model processes everything at once, it needs to know the order of words. Positional encoding adds "location tags" to the input embeddings.

4. Multi-Head Attention:
   The model looks at all the words in the sequence to figure out which ones are connected or important to each other. "Multi-Head" means it does this from several different perspectives simultaneously.

5. Add & Norm:
   The original input (before attention) is added back to the output (residual connection), and the result is normalized. This helps the network train effectively.

6. Feed Forward Network:
   The data passes through a standard neural network to pull out deeper meaning and complex patterns.

7. Add & Norm:
   Another residual connection and normalization step.

* Steps 4 through 7 repeat N times (Nx Blocks) to deeply encode the information before sending it to the Decoder.

<br>

**[  DECODER  ]**

1. Inputs (Shifted Right):
   The input to the decoder is the sequence of outputs generated so far. During training, the target sequence is shifted right to ensure the model learns to predict the next word based only on previous words, not future ones.

2. Output Embeddings:
   The input tokens are converted into high-dimensional vectors (numbers).

3. Positional Encoding:
   Just like in the encoder, "location tags" are added to the embeddings so the model understands the order of the words.

4. Masked Multi-Head Attention:
   This works like the encoder's self-attention, but it is "masked." The model is strictly blocked from looking ahead at future words in the sequence. It can only pay attention to the current and past words to figure out the context.

5. Add & Norm:
   The original input (before the masked attention) is added back to the output (residual connection), and the result is normalized.

6. Multi-Head (Cross-) Attention:
   This is the bridge between the Encoder and Decoder. The Decoder takes the rich, contextual information generated by the Encoder and compares it against its own current state. It decides which parts of the original input sequence it needs to focus on to generate the very next word.

7. Add & Norm:
   Another residual connection and normalization step.

8. Feed Forward Network:
   The data passes through a standard neural network to process complex patterns and deeper meaning.

9. Add & Norm:
   Another residual connection and normalization step.

* Steps 4 through 9 repeat N times (Nx Blocks) to refine the understanding before making a final prediction.

10. Linear Layer:
    After the blocks, the data passes through a Linear layer. This expands the output into a massive list of scores—one score for every single word in the model's entire vocabulary.

11. Softmax:
    This layer converts those raw scores into probabilities (percentages that add up to 100%).

12. Output Probabilities:
    The model looks at the probabilities, and the word with the highest percentage is selected as the predicted next word in the sequence.



```text
[ Input from Previous Layer / Embeddings ]
                                │
                                ▼
                     ┌─────────────────────┐
                     │                     │
                ┌────┤   Self-Attention    │
                │    │                     │
                │    └─────────┬───────────┘
                │              │
                │              ▼
                │    ┌─────────────────────┐
 (Residual) ────┼───►│     Add & Norm      │
                │    └─────────┬───────────┘
                │              │
                │              ▼
                │    ┌─────────────────────┐
                │    │                     │
                ├────┤ Feed Forward (FFN)  │
                │    │                     │
                │    └─────────┬───────────┘
                │              │
                │              ▼
                │    ┌─────────────────────┐
 (Residual) ────┼───►│     Add & Norm      │
                     └─────────┬───────────┘
                               │
                               ▼
                    [ Output to Next Block ]


--------------------------------------------------
HOW IT WORKS INSIDE THE BLOCK
--------------------------------------------------

1. Self-Attention (Understanding Context):
The model calculates the relationships between every single token (word) in the sequence simultaneously. It creates a weighted context vector for each word so the model understands how that word interacts with its surrounding neighbors (e.g., figuring out if "Apple" means the fruit or the tech company based on context).

2. First Add & Norm (Stabilizing the Signal):
- Add (Residual Connection): The original input to the attention layer is added directly back to its output. This "shortcut" prevents the network from losing crucial information as it gets deeper.
- Norm (Layer Normalization): The combined values are scaled to a standard range to keep mathematical values stable and speed up training.

3. Feed-Forward Network / FFN (Extracting Features):
While Self-Attention figures out *where* to look, the FFN figures out *what it all means*. It is a standard neural network that applies the exact same transformation to each token independently. It acts as a feature extractor to pull out deeper, more complex non-linear patterns.

4. Second Add & Norm (Final Polish):
The raw output of the FFN is added to its input (another residual connection) and normalized one last time.

Result:
The output is a highly refined, deeply contextualized matrix ready to be fed directly into the next identical block, or sent to the final output layers.
```





---



###### **ADITHYA UBALE**